## Requirements

In [24]:
from pathlib import Path

import lightning
lightning.pytorch.callbacks.early_stopping
import torch
import torch.nn
import torch.nn.functional
import torch.utils
import torchmetrics.classification 
import torchvision.datasets
import torchvision.transforms

## Data set

Define a directory to store the data.

In [2]:
data_dir = Path.cwd() / 'data'

The data should be converted to PyTorch tensors to be used for training.  It also helps to normalize the data so that the mean value of the images' pixels is 0.1307, and the standard deviation 0.3081.  This can be done using pre-defined transformers, composed into a pipeline.

In [3]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(
        (0.1307,),
        (0.3081,)
    ),
])

Now you can create a dataset for the training and the test data.

In [4]:
training_data = torchvision.datasets.MNIST(
    data_dir,
    train=True,
    download=True,
    transform=transform
)

In [5]:
training_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: /home/gjb/Projects/Machine-learning-with-Python/source-code/pytorch-lightning/data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.1307,), std=(0.3081,))
           )

In [6]:
training_size = int(0.8*len(training_data))
validation_size = len(training_data) - training_size

In [7]:
seed = torch.Generator().manual_seed(1234)
training_data, validation_data = torch.utils.data.random_split(
    training_data,
    [training_size, validation_size]
)

In [8]:
test_data = torchvision.datasets.MNIST(
    data_dir,
    train=False,
    download=True,
    transform=transform
)

In [9]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: /home/gjb/Projects/Machine-learning-with-Python/source-code/pytorch-lightning/data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.1307,), std=(0.3081,))
           )

## Data loaders

Although you already created the dataset objects, those should be wrapped in a data loader.  This ensures they are presented as batches during training, and are shuffled for each epoch.

In [10]:
train_loader = torch.utils.data.DataLoader(
    training_data,
    batch_size=256,
    num_workers=4,
)

In [11]:
validation_loader = torch.utils.data.DataLoader(
    validation_data,
    batch_size=256,
    num_workers=4
)

In [12]:
test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=256,
    num_workers=4
)

## Model

To use lightning, the network should be derived from `lightning.LightningModule`.  The network is constructed in the `__init__` function as an instance of `torch.nn.Sequential`.

Furthermore, the class should implement step methods:
* `training_step(...)`
* `validation_set(...)` (optional)
* `test_step(...)` (optional)

These methods each get a batch and its index as arguments.

Finally, the `configure_optimizers` method creates and configures the optimizer to use.

In [55]:
class MnistNet(lightning.LightningModule):

    def __init__(
        self,
        conv1_out_channels: int = 32,
        conv1_kernel_size: int = 3,
        conv2_out_channels: int = 64,
        conv2_kernel_size: int = 3,
        dropout1_p: float = 0.25,
        fc1_out_features: int = 128,
        dropout2_p: float = 0.5,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = torch.nn.Sequential(
            torch.nn.Conv2d(1, conv1_out_channels, conv1_kernel_size, 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(conv1_out_channels, conv2_out_channels, conv2_kernel_size, 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Dropout(dropout1_p),
            torch.nn.Flatten(1),
            torch.nn.LazyLinear(fc1_out_features),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout2_p),
            torch.nn.Linear(fc1_out_features, 10),
            torch.nn.LogSoftmax(dim=1),
        )

    def training_step(self, batch, batch_idx):
        x, y_target = batch
        y = self.model(x)
        loss = torch.nn.functional.nll_loss(y, y_target)
        self.log('training loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y_target = batch
        y = self.model(x)
        loss = torch.nn.functional.nll_loss(y, y_target)
        self.log('validation loss', loss)
        return loss

    def test_step(self, batch, batch_idx):
        x, y_target = batch
        y = self.model(x)
        loss = torch.nn.functional.nll_loss(y, y_target)
        self.log('test loss', loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1.0e-3)

## Training

Set the precision to control how tensor cores are utilized.

In [56]:
torch.set_float32_matmul_precision('medium')

Instantiate the neural network.

In [57]:
net = MnistNet()

Instatiate an early stopping criterion based on the validation loss.

In [58]:
early_stopping = lightning.pytorch.callbacks.early_stopping.EarlyStopping(
    monitor="validation loss",
    mode="min"
)

Create a trainer with the early-stopping criterion as a callback.

In [59]:
trainer = lightning.Trainer(
    callbacks=[early_stopping],
    max_epochs=200,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Train the network using the trainer with the training and validation dataloaders.

In [60]:
trainer.fit(
    model=net,
    train_dataloaders=train_loader,
    val_dataloaders=validation_loader
)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Sequential │ 20.1 K │ train │     0 │
└───┴───────┴────────────┴────────┴───────┴───────┘

Trainable params: 20.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 20.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Compute the loss on the test data.

In [61]:
trainer.test(net, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test loss         │   0.030453020706772804    │
└───────────────────────────┴───────────────────────────┘

[{'test loss': 0.030453020706772804}]

For each of the datasets, compute the accuracy.

In [62]:
data_loaders = {
    'training': train_loader,
    'valiation': validation_loader,
    'test': test_loader,
}

In [63]:
def compute_accuracy(model, device, data_loaders):
    accuracies = {}
    model.to(device)
    model.eval()
    for name, data_loader in data_loaders.items():
        accuracy = torchmetrics.classification.MulticlassAccuracy(
            num_classes=10
        ).to(device)
        with torch.no_grad():
            for x, y_target in data_loader:
                x, y_target = x.to(device), y_target.to(device)
                y = net.model(x)
                accuracy.update(y, y_target)
        accuracies[name] = accuracy.compute().item()
    return accuracies

In [64]:
for name, value in compute_accuracy(net, 'cuda', data_loaders).items():
    print(f'{name} accuracy: {value:.5f}')

training accuracy: 0.99897
valiation accuracy: 0.99145
test accuracy: 0.99175


The accuracy is quite high.